# DATA CLEANING AND FEATURE ENGINEERING

## SETUP

In [1]:
# import libraries
from pathlib import Path # paths
import duckdb # databank
import pandas as pd # data-wrangling
import numpy as np # data-wrangling
from functions import * # own functions

# set paths
PATH_BASE = Path.cwd().parent.parent
PATH_DATA = PATH_BASE / "data"
PATH_DB = PATH_DATA / "train.duckdb"
PATH_PARQUET = PATH_DATA / "train_delay_with_weather.parquet"

# connect to db
con = duckdb.connect(str(PATH_DB))

con.execute(f"""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('{PATH_PARQUET}')
            """)

# import data
df_raw = con.execute("SELECT * FROM train_delay").fetchdf()

## DATA INSPECTION

In [2]:
print(df_raw.info(), "\n")                      
print(df_raw.head(5), "\n")                      
print(df_raw["delay"].describe(), "\n")  
print(df_raw["train_type"].value_counts(), "\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2175996 entries, 0 to 2175995
Data columns (total 15 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   canceled           bool          
 6   arrival_planned    datetime64[us]
 7   arrival_real       datetime64[us]
 8   departure_planned  datetime64[us]
 9   departure_real     datetime64[us]
 10  delay              int32         
 11  temperature        float64       
 12  precipitation      float64       
 13  wind_speed         float64       
 14  weather_status     object        
dtypes: bool(1), datetime64[us](4), float64(3), int32(1), int64(1), object(5)
memory usage: 226.2+ MB
None 

   ride_id train_name train_type       station_current          station_dest  \
0    43233   ICE 1083        ICE  F

## DATA CLEANING

In [3]:
# filter train types (only ice and ic) 
df_cleaned = df_raw[df_raw["train_type"].isin(["ICE", "IC"])]

# remove canceled rides
df_cleaned = (
    df_cleaned.groupby("ride_id").filter(lambda g: not g["canceled"].any()))

# remove unnecessary columns
df_cleaned = df_cleaned.drop(columns=["weather_status", "canceled"])

# rename columns
df_cleaned = df_cleaned.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest"})

# rearange columns
df_cleaned = df_cleaned.loc[:, ["ride_id", "train_name", "train_type", "station_current", "station_dest",
               "departure_planned", "departure_real", "arrival_planned", "arrival_real",
                "temperature", "precipitation", "wind_speed", "delay"]]

# print results
print(df_cleaned.info())    

<class 'pandas.core.frame.DataFrame'>
Index: 1842266 entries, 0 to 2175995
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   departure_planned  datetime64[us]
 6   departure_real     datetime64[us]
 7   arrival_planned    datetime64[us]
 8   arrival_real       datetime64[us]
 9   temperature        float64       
 10  precipitation      float64       
 11  wind_speed         float64       
 12  delay              int32         
dtypes: datetime64[us](4), float64(3), int32(1), int64(1), object(4)
memory usage: 189.7+ MB
None


## NA ANALYSIS

In [4]:
# print number of na per column
print(df_cleaned.isnull().sum(axis = 0))

# missings in departure_planned and arrival_plannned
# are due to first or last station that do not have either arrival time (start station) or departure time (destination station)
len(df_cleaned["ride_id"].unique())

ride_id                   0
train_name                0
train_type                0
station_current           0
station_dest              0
departure_planned    200154
departure_real       200031
arrival_planned      200154
arrival_real         200036
temperature               0
precipitation             0
wind_speed                0
delay                     0
dtype: int64


200154

## CREATE FEATURES

### STATION START

In [5]:
# create station_start outside of function
# reason: also exists in 
# 
# 
#  REASON: ALSO EXISTS IN API DATA
### AFTER THAT, WE CAN APPLY FUNCTIONS TO BOTH DATASETS

# sorting
df_cleaned.sort_values(by=["ride_id", "departure_planned"])

# grouping
grouped = df_cleaned.groupby("ride_id")

# create station_start
df_cleaned["station_start"] = grouped["station_current"].transform("first")

### HISTORICAL FEATURES

In [6]:
# extract the historical features first (with function)
## station
hist_station = historic_delay_features(
    df_cleaned,
    variable = "station_current",
    prefix = "hist_delay_station")

## train name
hist_train = historic_delay_features(
    df_cleaned,
    variable = "train_name",
    prefix = "hist_delay_train")

# create a list for merging
historical_features_list = [hist_station, hist_train]

KeyboardInterrupt: 

### ALL OTHER FEATURES (WITH FUNCTION)

In [ ]:
df_features = create_features_historical(df = df_cleaned, historical_features = historical_features_list)

## GENERATE HISTORICAL DELAY LOOKUP FILE

In [ ]:
# get information on time frame
## last observed day
max_date = df_cleaned["departure_real"].max().floor("D")
## first day of time fram
window_start = max_date - pd.Timedelta("60D")
## select last 60 days
df_last60 = df_cleaned[(df_cleaned["departure_real"] > window_start)]

# aggregate data
## station
hist_delay_station = df_last60.groupby("station_current").agg(
    hist_delay_station_avg = ("delay", "mean"),
    hist_delay_station_q90 = ("delay", lambda x: x.quantile(0.9)),
    hist_delay_station_count = ("delay", "count")).reset_index()
## train 
hist_delay_train = df_last60.groupby("train_name").agg(
    hist_delay_train_avg = ("delay", "mean"),
    hist_delay_train_q90 = ("delay", lambda x: x.quantile(0.9)),
    hist_delay_train_count = ("delay", "count")).reset_index()

# store all in one lookup_list
hist_list_lookup = [hist_delay_station, hist_delay_train]

In [ ]:
# check if all stations are in lookup file
print("Number of stations in historical dataset:", len(df_cleaned["station_current"].unique()))
print("Number of stations in lookup file:", len(hist_list_lookup[0]))

# station names that are not in lookup file
stations_diff_list = list(set(df_cleaned["station_current"].unique()) - set(hist_list_lookup[0]["station_current"]))
stations_diff_list

Number of stations in historical dataset: 299
Number of stations in lookup file: 293


['Tübingen Hbf',
 'Bietigheim-Bissingen',
 'Heilbronn Hbf',
 'Berlin Friedrichstraße',
 'Berlin-Lichtenberg',
 'Kassel Hbf']

In [ ]:
# function: print number of rides and time frames (observed) for stations

def check_time_frame(df, station_name):
    
    # get data for station
    temp = df[df["station_current"] == station_name]

    # numbers of observations (= number of rides at station)
    n = len(temp)

    # get time_frame of observations
    t_arr_min, t_arr_max = temp["arrival_real"].min(), temp["arrival_real"].max()
    t_dep_min, t_dep_max = temp["departure_real"].min(), temp["departure_real"].max()

    print(f"Historical dataset contains {n} ride(s) for {station_name}")
    print(f"Time frame (arrivales):   {t_arr_min} - {t_arr_max}")
    print(f"Time frame (departures): {t_dep_min} - {t_dep_max}")
    print("")

# get rides and time frames for stations that are not inlcuded in lookup file
for station in stations_diff_list:
    check_time_frame(df_cleaned, station)

Historical dataset contains 207 ride(s) for Tübingen Hbf
Time frame (arrivales):   2024-07-01 21:43:00 - 2025-05-01 22:12:00
Time frame (departures): 2024-07-01 08:27:00 - 2025-05-02 08:13:00

Historical dataset contains 1 ride(s) for Bietigheim-Bissingen
Time frame (arrivales):   2024-10-19 22:01:00 - 2024-10-19 22:01:00
Time frame (departures): 2024-10-19 22:04:00 - 2024-10-19 22:04:00

Historical dataset contains 25 ride(s) for Heilbronn Hbf
Time frame (arrivales):   2024-07-19 12:15:00 - 2024-11-09 12:16:00
Time frame (departures): 2024-07-19 12:21:00 - 2024-11-09 12:21:00

Historical dataset contains 6 ride(s) for Berlin Friedrichstraße
Time frame (arrivales):   2024-08-08 23:23:00 - 2024-08-16 00:59:00
Time frame (departures): 2024-08-08 23:38:00 - 2024-08-16 00:59:00

Historical dataset contains 204 ride(s) for Berlin-Lichtenberg
Time frame (arrivales):   2024-07-26 22:44:00 - 2025-12-13 18:11:00
Time frame (departures): 2024-07-26 22:47:00 - 2025-10-29 17:44:00

Historical data

Explaining the results:
- Tübingen Hbf: not normally connected to the long-distance transport network, Riedbahn-renovation
- Berlin Lichtenberg: not normally connected to the long-distance transport network
- Heilbronn Hbf: not normally connected to the long-distance transport network, because of Riedbahn-renovation partly ICE trains (https://www.swr.de/swraktuell/baden-wuerttemberg/heilbronn/letzter-ice-halt-in-heibronn-100.html)
- Berlin Friedrichstraße: not normally connected to the long-distance transport network
- Kassel Hbf: long-distance trains usually go via Kassel-Wilhelmshöhe
- Bietigheim-Bissingen: not normally connected to the long-distance transport network

In [ ]:
# check if all trains are in lookup file
print("Number of trains in historical dataset:", len(df_cleaned["train_name"].unique()))
print("Number of trains in lookup file:", len(hist_list_lookup[1]))

# train names that are not in lookup file (not printed because list too long)
trains_diff_list = list(set(df_cleaned["train_name"].unique()) - set(hist_list_lookup[1]["train_name"]))

Number of trains in historical dataset: 1340
Number of trains in lookup file: 960


## EXPORT DATA

In [ ]:
# cleaned features
df_features.to_parquet('data/train_delay_with_features.parquet', index=False)

# lookup tables
## stations
hist_list_lookup[0].to_parquet("data/hist_delay_station_lookup.parquet", index=False)
## trains
hist_list_lookup[1].to_parquet("data/hist_delay_train_lookup.parquet", index=False)